# Confidence of a segmentation

In [ ]:
#| include: false
#| echo: false
# Setup cell: makes the notebook work in Jupyter *and* in both Quarto builds.
import os, sys
from pathlib import Path

if "COURSE_LECTURE_DIR" not in os.environ:          # interactive use
    os.environ["COURSE_LECTURE_DIR"] = str(Path.cwd())
    sys.path.insert(0, str(Path.cwd().parents[1] / "scripts"))

import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
import skimage.filters as flt
import pandas as pd
import matplotlib.patches as patches
import tifffile as tiff
from lecture8_support import *
import confmap as cm
import sys
sys.path.append('../common/')
# import plotsupport as ps
import pointcloud as pc
from IPython.display import Markdown, display

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

dcolors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%matplotlib inline
%config InlineBackend.figure_format = 'retina' # This is get sharper plots on digh density screens like the Retina screen on Macs

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False})

## Confidence map without ground truth

_Freely from P. Moonen's talk at ICTMS2019_

There are different ways to evaluate segmentation algorithm performance;

- Confusion matrix
- ROC curve 
- Hit map

... but they require a ground truth

What if we want to know how well an image was segmented without ground truth?

## Assume Gaussian noise

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Histogram of noise in the image."
#| label: fig-noisehist
s0=1;
noise = np.random.normal(size=[200,200])
fig,ax = plt.subplots(1,2,figsize=(12,5))
ax[0].imshow(noise,cmap='gray')
h,bins = np.histogram(noise.ravel(),bins=100)
normh=h/h.sum()
ax[1].fill(bins[:-1],normh,alpha=0.5,label='Noise histogram')
x=np.linspace(-5,5,201)
pdf = cm.gaussian(x,0,1)
ax[1].plot(x,1.5*pdf/pdf.sum(),color=dcolors[1], label='Gaussian pdf')
ax[1].legend();
plt.show()

***
### Threshold between two Gaussian classes
_From lecture 4_

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Thresholding of a noisy image."
#| label: fig-thresholdingnoisy
x=np.linspace(-6,6,1000)
h0 = np.exp(-(x-1)**2/2)
h1 = np.exp(-(x+1)**2/2)

gamma=0.5

# Visualization
gidx = np.abs(x - gamma).argmin()
plt.fill(x,h0,label='True positive',alpha=0.3,ec=dcolors[0],lw=2)
plt.fill(x,h1,label='True negative',alpha=0.3,ec=dcolors[1],lw=2)

plt.vlines([x[gidx]],ymin=0,ymax=1,color='magenta',label='Threshold $\gamma$={0}'.format(gamma))

plt.fill_between(x[:gidx],0,h0[:gidx],color=dcolors[2],label='False negative',alpha=0.3,ec=dcolors[2],lw=2)
plt.fill_between(x[gidx:],0,h1[gidx:],color=dcolors[3],label='False positive',alpha=0.3,ec=dcolors[3],lw=2)

plt.legend();
plt.show();

***
### Compute the confidence based on the class pdf's

We want the probability mix of the classes for each gray level:

1. Find the pdf for each class using Gaussian Mixture Models ([GMM](https://scikit-learn.org/stable/modules/mixture.html))
2. Normalize
3. For each gray level $x$ compute weights
$$w_i(x)= pdf_i(x)$$
for Gaussians $w_i(x)=e^{-\frac{(x-\mu_i)^2}{2 \sigma_i^2}}$
4. Then the confidence for class $i$ is 
$$c_i(x)=\frac{w_i(x)}{\sum_j w_j(x)}$$
5. Create a confidence map using the segmented image to select $c_i$ for each pixel

***
### What are the weights?

In our algorithm we used $w_i(x)=e^{-\frac{(x-\mu_i)^2}{2 \sigma_i^2}}$

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Computing the weights."
#| label: fig-computingweights
x=np.linspace(-5,5,1000)
g0=cm.gaussian(x,-1,1)
g0m=g0.max()
g0=g0/g0.max()
g1=cm.gaussian(x,1,1)
g1m=g1.max()
g1=g1/g1.max()
_,ax=plt.subplots(1,figsize=(10/2.54,8/2.54))
ax.plot(x,g0,label=r'Class 0')
ax.plot(x,g1,label=r'Class 1')
xx=0.5
arrow = patches.FancyArrowPatch((xx,0), (xx,cm.gaussian(xx,1,1)/g1m),
                                 mutation_scale=20,
                                 color=dcolors[3],
                                 arrowstyle='-|>',linewidth=2,
                                label=r'$w_1$'
                                )
ax.add_patch(arrow);

arrow = patches.FancyArrowPatch((xx,0), (xx,cm.gaussian(xx,-1,1)/g0m),
                                 mutation_scale=20,
                                 color=dcolors[2],
                                 arrowstyle='-|>',linewidth=2,
                                label=r'$w_0$'
                                )
ax.add_patch(arrow);

ax.text(4,0.5,r'$c_0=\frac{w_0}{w_0+w_1}=\frac{0.3246}{0.3246+0.8825}=0.2689$',size=12)
ax.text(4,0.25,r'$c_1=\frac{w_1}{w_0+w_1}=\frac{0.8825}{0.3246+0.8825}=0.7311$',size=12)

plt.legend()
plt.axis('off');
plt.show();

***
### Example: Three phase segmentation

Our test data

- Three classes with $\mu$=[-3,1,5]
- Gaussian noise $~ \mathcal{N}(\mu, \sigma=1.0)$
- Unsharpness - Gaussian filter ($\sigma$=1)

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Image with three classes."
#| label: fig-threeclasses
ss = 1
N = 20 
m=np.array([-3,1,5])
s=np.array([ss,ss,ss])
xx=np.linspace(-10,10,1001)
res = np.array([cm.multi_gaussian(x,m=m,s=s) for x in xx])

img=np.repeat(np.repeat(np.array([[m[1],m[0]],[m[0],m[2]]]),N,axis=0),N,axis=1)
seg=np.zeros(img.shape)
fimg = flt.gaussian(img,sigma=1,preserve_range=True)
for idx in range(len(m)):
    seg[img==m[idx]]=idx

nm = fimg+np.random.normal(loc=0,scale=ss,size=img.shape)

cmap = cm.conf_map(nm,seg,m,s)

fig,ax=plt.subplots(1,3,figsize=(1.25*15/2.54,1.25*5/2.54))
ax=ax.ravel()
ax[0].imshow(seg,cmap='gray')
ax[0].set_title('Ground truth/segmentation')
ax[1].imshow(fimg,cmap='gray')
ax[1].set_title('Smooth image')
ax[2].imshow(nm,cmap='gray')
ax[2].set_title('Noisy image');
plt.show();

***
### The confidence maps for the segmenation
_Here, we cheat by using the original class image as segmented image_

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Segmentation results with confidence maps."
#| label: fig-segmentationresults
seg=np.zeros(img.shape)
fimg = flt.gaussian(img,sigma=1,preserve_range=True)
for idx in range(len(m)):
    seg[img==m[idx]]=idx

fig,ax=plt.subplots(2,4,figsize=(15,8))

ax=ax.ravel()
ax[0].imshow(nm,cmap='gray')
ax[0].set_title('Noisy image')

for idx in range(3):
    cmap = cm.conf_map(nm,seg,m,s,c=idx)
    ax[idx+1].imshow(cmap,clim=[0,1])
    ax[idx+1].set_title('Confidence class {}'.format(idx))

ax[4].hist(nm[seg==0].ravel(),bins=40, label="Class $\mu$=-3", alpha=0.3);
ax[4].hist(nm[seg==1].ravel(),bins=40, label="Class $\mu$=1", alpha=0.3);
ax[4].hist(nm[seg==2].ravel(),bins=40, label="Class $\mu$=5", alpha=0.3);
ax[4].legend()

# ax[4].plot(xx,res);
# ax[4].set_title('Normalized class distributions')

cmap = cm.conf_map(nm,seg,m,s)
ax6=ax[6].imshow(cmap,clim=[0,1])
ax[6].set_title('Confidence map')
xx,w = cm.conf_plot(m,s)
for idx in range(len(m)) :
    ax[5].plot(xx,w[idx]/np.sum(w,axis=0), label="C {}".format(idx));
    
ax[5].legend()
ax[5].set_title('Confidences');

ax[7].axis('off')
fig.colorbar(ax6,ax=ax[7],location='bottom');
plt.show();

## When does it matter?

:::: {.columns}
::: {.column width=50%}
        
### High SNR images
Edges have lowest confidence; most impact for

- Perimeter
- Turtosity
- Network connectivity
- Volumes of small items
        
:::
::: {.column width=50%}
            
### Low SNR image
Low confidence on most pixels!

- All high-SNR issues
    
and
    
- Porosity
- Volumes
- Labeling
            
:::
::::

## Concluding uncertainties

- Scientific analysis requires uncertainties

- Uncertainties in imaging is not straight forward
    - Physical uncertainties can be quantified
    - Noise can be measured
    - Image analysis precision depends on object/pixel size ratio
    - The effect of processing must be quantified through simulations